# Sum-of-squares relaxation: validity of a single hex

Hex meshes are everywhere in simulation, but checking that a single hexahedral element is *valid*, meaning no flipped or degenerate Jacobian inside the element, is a non-convex polynomial optimization problem. 

The Jacobian determinant of a trilinear hex map is a polynomial of total degree six in three variables, and we want
$$ \min_{(u, v, w) \in [0, 1]^3} J(u, v, w). $$
If this minimum is positive, the hex is valid. If it is negative, the element is folded somewhere inside.

Direct approaches like sample-on-a-grid are unreliable, and gradient descent gets stuck in spurious local minima. Marschner, Palmer, Zhang, Solomon (2020) show this is exactly the kind of problem where sum-of-squares (SOS) relaxation shines. By Putinar's positivstellensatz,
$$ J(u, v, w) - \beta \;=\; s_0(u, v, w) + s_1(u, v, w) \, u + s_2 (1 - u) + s_3 v + s_4 (1 - v) + s_5 w + s_6 (1 - w), $$
with each $s_i$ a sum of squares iff $J(u, v, w) \ge \beta$ on the unit cube. Maximising $\beta$ over all such SOS decompositions is an SDP. Each $s_i = m_i^\top Q_i m_i$ where $m_i$ is a fixed monomial basis and $Q_i \succeq 0$, so the SOS constraints become matrix PSD constraints.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys, pathlib
sys.path.append(str(pathlib.Path('..').resolve()))

import numpy as np
import sympy as sp
import matplotlib.pyplot as plt

from src.sos_symbolic_jacobian_det import sos_symbolic_jacobian_det
from src.sos_jacobian_single_hex import sos_jacobian_single_hex

## A valid hex: a small perturbation of the unit cube

We start with the standard unit cube and perturb each corner by a small Gaussian. Small enough perturbations keep the element valid, meaning the Jacobian stays positive everywhere.

In [ ]:
# Standard hex corner ordering used by Marschner et al.
V_cube = np.array([[0, 0, 0], [1, 0, 0], [1, 1, 0], [0, 1, 0],
                    [0, 0, 1], [1, 0, 1], [1, 1, 1], [0, 1, 1]], dtype=float)
rng = np.random.default_rng(0)
V_valid = V_cube + rng.normal(0, 0.1, V_cube.shape)

bound, argmin, recovery_gap = sos_jacobian_single_hex(V_valid, k=4)
print(f'SOS lower bound on min J: {bound:.6f}')
print(f'Recovered argmin: {argmin}')
print(f'J(argmin) - bound: {recovery_gap:.2e} (near 0 = relaxation is tight)')

The bound is positive, so the perturbed cube is valid. The recovery gap `J(argmin) - bound` is at numerical zero, which means the SOS relaxation is *tight*: the lower bound on $\min J$ equals the true minimum, and the recovered point $(u^*, v^*, w^*)$ really is the argmin. We can sanity-check that with a brute force grid search.

In [ ]:
u_sym, v_sym, w_sym = sp.symbols('u v w')
J_poly = sos_symbolic_jacobian_det(V_valid)
J_func = sp.lambdify((u_sym, v_sym, w_sym), J_poly.as_expr(), 'numpy')

print(f'J at recovered argmin : {float(J_func(*argmin)):.6f}')
print(f'SOS lower bound : {bound:.6f}')

# Sanity check: the brute force minimum over a fine grid
us = np.linspace(0, 1, 51)
U, Vg, Wg = np.meshgrid(us, us, us, indexing='ij')
vals = J_func(U, Vg, Wg)
print(f'Brute force min on grid: {vals.min():.6f}')

## An invalid hex: pushing one corner inside the cube

Now we make the element clearly invalid by moving the top-front-right corner so it lies *inside* the cube. The Jacobian should become negative somewhere, and the SOS bound should be negative.

In [ ]:
V_invalid = V_cube.copy()
V_invalid[6] = np.array([0.4, 0.4, 0.4])

bound, argmin, recovery_gap = sos_jacobian_single_hex(V_invalid, k=4)
print(f'SOS lower bound on min J: {bound:.6f}')
print(f'Recovered argmin: {argmin}')
print(f'J(argmin) - bound: {recovery_gap:.2e}')

J_poly = sos_symbolic_jacobian_det(V_invalid)
J_func = sp.lambdify((u_sym, v_sym, w_sym), J_poly.as_expr(), 'numpy')
vals = J_func(U, Vg, Wg)
print(f'Brute force min on grid: {vals.min():.6f}')

Negative bound, negative grid minimum, recovery gap still near zero. The relaxation correctly certifies the element is invalid and hands us the location where the Jacobian is most negative. The whole certification took one SDP.

## Visualization

In [ ]:
import polyscope as ps

# The six quad faces of a hex in the standard corner ordering
hex_quads = np.array([[0, 1, 2, 3],   # bottom (w = 0)
                        [4, 5, 6, 7],  # top (w = 1)
                        [0, 1, 5, 4],  # front (v = 0)
                        [3, 2, 6, 7],  # back (v = 1)
                        [0, 3, 7, 4],  # left (u = 0)
                        [1, 2, 6, 5],  # right (u = 1)
                        ], dtype=np.int64)
hex_tris = np.vstack([hex_quads[:, [0, 1, 2]], hex_quads[:, [0, 2, 3]]])

def hex_corner_jacobians(V):
    """Sample J at each of the eight corners. Parameter values 0/1 per axis."""
    J_poly = sos_symbolic_jacobian_det(V)
    J_func = sp.lambdify((u_sym, v_sym, w_sym), J_poly.as_expr(), 'numpy')
    corner_params = np.array([[0, 0, 0], [1, 0, 0], [1, 1, 0], [0, 1, 0],
                            [0, 0, 1], [1, 0, 1], [1, 1, 1], [0, 1, 1]], dtype=float)
    return np.array([float(J_func(*c)) for c in corner_params])

ps.init()
ps.remove_all_structures()
ps.set_ground_plane_mode('none')

# Visualization
shift = np.array([1.5, 0.0, 0.0])
ps_valid = ps.register_surface_mesh('valid hex', V_valid, hex_tris, smooth_shade=False)
ps_invalid = ps.register_surface_mesh('invalid hex', V_invalid + shift, hex_tris, smooth_shade=False)
ps_valid.add_scalar_quantity('J at corners', hex_corner_jacobians(V_valid), cmap='coolwarm', enabled=True)
ps_invalid.add_scalar_quantity('J at corners', hex_corner_jacobians(V_invalid), cmap='coolwarm', enabled=True)
ps.show()

**▶ Your turn.** Re-run the invalid example with smaller and smaller perturbations of the bad corner (e.g. `V_invalid[6] = np.array([0.95, 0.95, 0.95])`, then `[0.85, 0.85, 0.85]`, and so on...) and watch the SOS bound cross zero. The exact crossover is the boundary between valid and invalid elements, and SOS finds it without any sampling. With `k=4` the relaxation is tight on essentially every random hex. Bumping `k` higher only matters for adversarial corner cases.

## References

[1] *Sum-of-squares geometry processing*.
       Marschner, Z.; Palmer, D.; Zhang, P.; Solomon, J.
       ACM Transactions on Graphics (SIGGRAPH Asia) 40 (6): 1 (2021).
       :DOI:`10.1145/3478513.3480551`